# 03 — Single-Position Deep Dive

**Notebook 3 of the *Developer Guide to Disciplined Trading* series.**

> Prerequisites: [`01-foundations`](./01-foundations-techtrade-and-analysis.ipynb), [`02-morning-scan`](./02-morning-scan.ipynb). You should already have run a scan and have a candidate ticker in mind.

---

## Alex's question this notebook answers

> *"The morning scan flagged NVDA. Should I take the trade — and if I do, what does the engine think the position should look like at the bar-by-bar level?"*

Notebook 02 produced a ranked list across all 11 sectors. **This notebook zooms in on ONE name** and shows the full single-position pipeline that produces it:

1. **`signals`** — the per-symbol confluence score + the indicator vote attribution (read "why this score?")
2. **`plan`** — turn the signal into a complete `TradePlan` with entry / stop / target levels, position size, and an inline `Recommendation`
3. **`orders`** — materialize the broker-ready order legs (entry + stop + target + time-exit)
4. **`simulate`** — paper-fill those order legs against a forward bar window. **No look-ahead** — bar-`t` signals only fill at `t+1`.

## What Alex takes away

By the end he can answer:

- **What does each order leg do?** (entry vs exit_stop vs exit_target vs exit_time)
- **Where would today's signal have actually filled?** (next-bar-open with slippage + commission)
- **Which exit triggered first** on the simulated path (the answer is *not always* the one Alex would have guessed).
- **How much would a wider stop have cost in position size** (the risk-budget-vs-stop-distance trade-off, made concrete).

## Wall-clock

~3-5 min warm. Most of it is the OHLCV fetch for the chosen symbol; the engine math is sub-second.

## Choose your ticker

We use `NVDA` as the running example because it's liquid, well-known, and almost always has a non-flat signal. **Swap `SYMBOL` below** for any ticker the scan flagged for you. If you don't have one yet, run notebook 02 first.

In [ ]:
# Edit this and re-run the rest of the notebook.
SYMBOL = "NVDA"
PRESET = "trend_follow"  # try "mean_revert" or "breakout" for the same ticker — surprising differences
RISK = 0.01              # 1% of notional per trade

## 1. Setup probe

In [ ]:
from openbb import obb
from datetime import date
print(f"obb loaded; today is {date.today().isoformat()}")
print(f"Studying {SYMBOL} under preset='{PRESET}' at risk={RISK:.1%}")

## 2. The raw signal — `obb.techtrade.signals`

Before any plan or order assembly, just compute the confluence score for this one ticker. `signals` is the lightest call in the pipeline: it builds the indicator panel, runs the confluence engine under the chosen preset, returns the score + the vote attribution. No sizing, no orders.

Wall-clock: ~3-8 seconds (one OHLCV fetch + indicator math).

In [ ]:
sigs = obb.techtrade.signals(symbols=[SYMBOL], preset=PRESET).results
if not sigs:
    raise RuntimeError(f"No signal for {SYMBOL} — maybe the symbol is unsupported on fmp_cached.")
sig = sigs[0]
print(f"{sig.symbol}  as_of={sig.as_of}  segment={sig.segment}")
print(f"  composite score:  {sig.score:+.4f}  direction: {sig.direction}")
print(f"  vote count:       {len(sig.votes)}")

### Read the votes

Every indicator that contributed to the score is shown with its `family / name / vote / weight`. The composite is the weighted sum (with the volume family acting as a multiplier, not an additive term — see PRD §12.2). The cell below sorts by absolute contribution so the loudest voters are at the top.

In [ ]:
import pandas as pd

votes_df = pd.DataFrame([
    {"family": v.family, "name": v.name, "vote": v.vote, "weight": v.weight, "contrib": v.vote * v.weight}
    for v in sig.votes
])
votes_df = votes_df.reindex(votes_df["contrib"].abs().sort_values(ascending=False).index).reset_index(drop=True)
votes_df

### What to look for

- **Sign agreement across families.** If trend + momentum + volatility all vote `+`, the score is the result of confluence not coincidence.
- **A high-weight family voting against.** Weight 0.40 on trend; if trend votes `-1.0` while every other family votes `+`, the composite might still cross the threshold but the strongest voice is dissenting. That's a setup Alex doesn't take.
- **Volume.** Volume votes are amplification, not addition. Negative volume votes (distribution) damp the score; positive (accumulation) amplify.

## 3. The full plan — `obb.techtrade.plan`

`signals` answered *should I trade?* `plan` answers *if I trade, what's the trade?* The plan adds:

- **Levels**: entry / stop / target prices (sized off ATR(14) and the rule's `atr_stop_mult` + `target_r_multiple`)
- **Sizing**: `position_size` in shares such that the stop-distance × shares ≈ `risk` × notional
- **Orders**: broker-ready order legs with `intent` tags (`entry` / `exit_stop` / `exit_target` / `exit_time` / `exit_signal`)
- **Recommendation**: human-facing summary (`action`, `conviction`, `reasoning`, `caveats`)
- **`validation`**: empty until notebook 04 fills it via `validate`

In [ ]:
plans = obb.techtrade.plan(symbols=[SYMBOL], preset=PRESET, risk=RISK).results
if not plans:
    raise RuntimeError(f"plan() returned no plans for {SYMBOL} — score below entry_threshold?")
plan = plans[0]
rec = plan.recommendation

print(f"--- TradePlan for {plan.symbol} ({plan.segment}, as_of {plan.as_of}) ---\n")
print(f"  Action:        {rec.action}  ({rec.conviction})")
print(f"  Score:         {plan.signal.score:+.4f}")
print(f"  Entry price:   ${rec.entry_price}")
print(f"  Stop price:    ${rec.stop_price}  ({rec.stop_distance_pct*100:.2f}% from entry)")
print(f"  Target price:  ${rec.target_price}  ({rec.target_distance_pct*100:.2f}% from entry)")
print(f"  R:R:           {rec.risk_reward:.2f}")
print(f"  ATR(14):       {rec.atr:.2f}")
print(f"  Position size: {rec.position_size} shares")
print(f"  Risk/share:    ${rec.risk_per_share}")
print(f"  Risk % notnl:  {rec.risk_pct_of_notional*100:.3f}%")
print(f"  Time stop:     {rec.time_stop_bars} bars")
print(f"\n  Caveats: {rec.caveats}")

### The sizing math

`position_size = floor(risk_budget / risk_per_share)` where:

- `risk_budget = risk × notional` (you set `risk` — `notional` defaults to the engine's convention; see PRD §13)
- `risk_per_share = entry_price - stop_price` for longs (or `stop - entry` for shorts)

This means **a wider stop = smaller position**. The engine does the inverse for you. The cell below makes the trade-off concrete.

In [ ]:
from decimal import Decimal

entry = float(rec.entry_price)
stop = float(rec.stop_price)
atr = rec.atr
qty = float(rec.position_size)

# What if the rule used a tighter or wider ATR multiplier?
# The engine default is atr_stop_mult=2.0 (rule.atr_stop_mult).
print(f"At default 2.0×ATR stop: stop {stop:.2f}, distance {abs(entry-stop):.2f} ({abs(entry-stop)/entry*100:.2f}%), qty {qty:.0f}")
for mult in (1.0, 1.5, 2.5, 3.0):
    sim_stop_dist = mult * atr
    scaled_qty = qty * (2.0 / mult)  # qty scales as risk/distance → 1/mult
    print(f"  hypothetical {mult}×ATR: distance {sim_stop_dist:.2f} ({sim_stop_dist/entry*100:.2f}%), qty would be {scaled_qty:.0f}")

## 4. Materialize the orders — `obb.techtrade.orders`

The plan carries the orders inline. `obb.techtrade.orders(plan=plan)` is the idempotent re-validation that confirms the plan round-trips cleanly through a JSON boundary (which matters when plans are exported to Excel or sent across a network). Returns the same `list[Order]`.

In [ ]:
orders = obb.techtrade.orders(plan=plan).results
print(f"--- {len(orders)} order leg(s) for {plan.symbol} ---\n")
for o in orders:
    parts = [
        f"intent={o.intent:<12}",
        f"side={o.side:<10}",
        f"qty={o.quantity}",
        f"type={o.order_type}",
    ]
    if o.limit_price is not None:
        parts.append(f"limit=${o.limit_price}")
    if o.stop_price is not None:
        parts.append(f"stop=${o.stop_price}")
    parts.append(f"tif={o.tif}")
    print("  " + " ".join(parts))

### What each leg means

- **`entry`** is the first leg — typically a market order at `t+1` open (no look-ahead).
- **`exit_stop`** is the protective stop. If price trades through the stop, this leg fires and closes the position at-market.
- **`exit_target`** is the take-profit limit. Closes the position when price reaches the target.
- **`exit_time`** is the time stop (default `max_holding_bars=20` business days). If neither stop nor target has fired by then, close at-market.
- **`exit_signal`** (not always present) fires if the composite confluence flips to the opposite side mid-trade.

A real broker integration knows which leg is which from the `intent` tag — it's the contract between techtrade and any downstream execution layer.

## 5. Paper-fill the orders — `obb.techtrade.simulate`

`simulate` walks the order legs forward over a bar window and returns the realized fills. The fill model:

- **Bar-`t` signals fill at `t+1` open.** This is the no-look-ahead discipline. The engine refuses to fill on the bar that produced the signal.
- **Slippage** is applied per-fill (default convention; see `engine/execution.py`).
- **Commission** is applied per-fill (configurable).
- **The first exit to trigger wins.** If price hits the stop before the target, you get the stop-fill and the target leg is canceled.

Below we fetch a real forward window (the trailing 30 business days for the chosen symbol) and replay the orders over it.

In [ ]:
# Pull a recent OHLCV window — the forward bars the simulator will walk through.
# Wall-clock: 2-5s cached.
from datetime import date, timedelta
end = date.today()
start = end - timedelta(days=45)  # ~30 business days

ohlcv_obj = obb.equity.price.historical(
    symbol=SYMBOL,
    start_date=str(start),
    end_date=str(end),
    provider="fmp_cached",
)
ohlcv_df = ohlcv_obj.to_dataframe()
print(f"Fetched {len(ohlcv_df)} bars for {SYMBOL} from {start} to {end}.")
ohlcv_df.tail(5)

In [ ]:
# techtrade expects bars in a particular shape (per the README §Commands signature for simulate).
# Convert the OHLCV dataframe into the list-of-dict shape simulate consumes.
from decimal import Decimal

def _to_decimal(x):
    return Decimal(str(round(float(x), 4)))

bars = []
for ts, row in ohlcv_df.iterrows():
    bars.append({
        "symbol": SYMBOL,
        "timestamp": ts.isoformat() if hasattr(ts, "isoformat") else str(ts),
        "open":   _to_decimal(row["open"]),
        "high":   _to_decimal(row["high"]),
        "low":    _to_decimal(row["low"]),
        "close":  _to_decimal(row["close"]),
        "volume": _to_decimal(row["volume"]),
    })
print(f"Prepared {len(bars)} bars for simulate.")
print("First bar:", {k: bars[0][k] for k in ('symbol','timestamp','close')})
print("Last bar:",  {k: bars[-1][k] for k in ('symbol','timestamp','close')})

In [ ]:
# Run the paper-broker forward over the bar window.
fills = obb.techtrade.simulate(orders=orders, bars=bars).results
print(f"--- {len(fills)} paper fill(s) ---\n")
for f in fills:
    print(
        f"  ts={f.timestamp}  {f.side:<10} qty={f.quantity}  price=${f.price}  "
        f"slippage=${f.slippage}  commission=${f.commission}"
    )

### What the fills tell Alex

- **An entry fill** at `t+1` open — confirms the no-look-ahead discipline (the signal was at bar `t`, fill is at the NEXT bar).
- **An exit fill** (stop, target, or time) — this is what *actually happened* on the historical path. The exit's `intent` is preserved in the order it came from; cross-reference by matching `order_ref` to the orders list.
- **Slippage** non-zero — the engine assumed the price slipped on entry; that's a real-world cost most paper backtests ignore.
- **0 fills** — possible: if the entry never triggered (e.g. for a limit entry that never reached the limit) OR the bar window starts AFTER the signal date. This is realistic; not a bug.

## 6. Realized P&L on the simulated path

If at least one entry + one exit fill exist, Alex can compute what the trade *actually returned* on the historical window. Compare it to what the recommendation *predicted*.

In [ ]:
if not fills:
    print("No fills — no realized P&L to compute. Try a longer bar window.")
else:
    entries = [f for f in fills if f.side in ("buy", "sell_short")]
    exits = [f for f in fills if f.side in ("sell", "buy_to_cover")]
    if not (entries and exits):
        print("Have entry but no exit yet (position still open in the window), or vice versa.")
    else:
        entry_fill = entries[0]
        exit_fill = exits[0]
        qty = float(entry_fill.quantity)
        ep = float(entry_fill.price)
        xp = float(exit_fill.price)
        slippage_total = float(entry_fill.slippage) + float(exit_fill.slippage)
        commission_total = float(entry_fill.commission) + float(exit_fill.commission)
        if entry_fill.side == "buy":
            gross = (xp - ep) * qty
        else:  # sell_short -> buy_to_cover
            gross = (ep - xp) * qty
        net = gross - slippage_total - commission_total
        notional = ep * qty
        print(f"  entry: ${ep:.2f} × {qty:.0f} = notional ${notional:,.2f}")
        print(f"  exit:  ${xp:.2f}  ({exit_fill.timestamp})")
        print(f"  gross P&L:       ${gross:+,.2f}")
        print(f"  - slippage cost: ${slippage_total:.2f}")
        print(f"  - commission:    ${commission_total:.2f}")
        print(f"  = NET P&L:       ${net:+,.2f}  ({net/notional*100:+.2f}% of notional)")

### What this number means

**It does NOT mean the trade was good.** A single sample of one historical path is not statistical evidence. The number tells Alex *what happened on this one walk-forward*. The whole point of notebook 04 (`validate`) is to do this thousands of times over many resampled folds and compute a PBO + DSR — turning a single anecdote into a statistic.

But the single sample IS useful for two things:

1. **Sanity-check the engine.** If the rec said BUY and the simulate said "hit stop before target", that's a coherent loss. If the simulate produced something nonsensical (entry without an exit, fills at the same bar as signal), there's a bug.
2. **Feel the cost of friction.** Slippage + commission together can easily eat 20-50% of a small-edge trade. Seeing it in dollars makes Alex re-think a 1.5 R:R setup.

## 7. Variation: same symbol, different preset

Re-run the WHOLE chain (signal → plan → simulate) under the other two presets to see how the trade shape changes. Trend-followers and mean-reverters disagree about this exact ticker on this exact day.

In [ ]:
comparison = []
for p_name in ("trend_follow", "mean_revert", "breakout"):
    p_plans = obb.techtrade.plan(symbols=[SYMBOL], preset=p_name, risk=RISK).results
    if not p_plans:
        comparison.append({"preset": p_name, "action": "(no plan)", "score": None, "r:r": None, "qty": None})
        continue
    pp = p_plans[0]
    comparison.append({
        "preset": p_name,
        "score": round(pp.signal.score, 4),
        "action": pp.recommendation.action,
        "conviction": pp.recommendation.conviction,
        "entry": float(pp.recommendation.entry_price),
        "stop": float(pp.recommendation.stop_price),
        "target": float(pp.recommendation.target_price),
        "r:r": round(pp.recommendation.risk_reward, 2),
        "qty": float(pp.recommendation.position_size),
    })

pd.DataFrame(comparison)

### What disagreement looks like

- **All three agree on direction** (all BUY or all SELL_SHORT) and roughly on entry/stop/target — strong signal. The presets emphasize different facets but the underlying setup is robust.
- **Two agree, one is FLAT** — the dissenting preset finds insufficient signal under its weighting. Read the dissenter's votes (run cell 2 again with that preset) to see WHICH family is dragging the composite down.
- **Split signals (e.g. trend_follow says BUY, mean_revert says SELL_SHORT)** — Alex usually stays out. He's looking at a name where the bull/bear case is balanced; neither side has an edge.

## 8. The single-position checklist

Before Alex takes a real position on the symbol he studied here:

- [ ] **`score` is decisively past the entry threshold** (not just barely; he wants `|score| >= 0.5` for personal margin).
- [ ] **`r:r >= 2.0`** so the math works even at his historical hit rate (~45%).
- [ ] **The audit trail (votes) shows family agreement** — not just one indicator at full weight.
- [ ] **The simulated forward window's exit makes sense** — stop hits before target on losers, target before stop on winners, no fills-on-same-bar anomalies.
- [ ] **The trade has been through `validate`** (notebook 04). A single backtest is anecdote; a PBO + DSR + verdict is statistical evidence.

If any of those fail, **skip the trade** or re-run with a different preset / different ticker. 

---

## What's next

- **Notebook 04 — The Validation Gate**: take this exact plan and call `obb.techtrade.validate(plan, method="wfo", horizon_years=5)`. PBO + DSR + verdict. The anti-overfit gate that turns the single simulated path above into statistical evidence.
- **Notebook 05 — Per-Sector Tuning**: would different indicator periods have given a different score on this ticker? `obb.techtrade.tune(segment=...)` proposes new periods per-sector; persists only the ones that pass `validate`.
- **Notebook 06 — Audit and Replay**: cross-reference what Alex actually did with what the engine suggested. Journal entry. End-of-day discipline.

---

*End of notebook 03.*